In [1]:
# =====================================
# CONSTRUCCIÓN DE LA CAPA GOLD
# Consolidación de restaurantes enriquecidos en tabla única
# =====================================

import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

# Rutas
PROCESSED_DIR = Path("../data/processed")
GOLD_DIR = Path("../data/gold")
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print("Librerías cargadas y rutas configuradas")
print(f"Processed: {PROCESSED_DIR.resolve()}")
print(f"Gold: {GOLD_DIR.resolve()}")

Librerías cargadas y rutas configuradas
Processed: /Users/juana/Desktop/tfm-data-science-gtm/data/processed
Gold: /Users/juana/Desktop/tfm-data-science-gtm/data/gold


In [2]:
# =====================================
# CARGAR FICHEROS DE PROCESSED
# =====================================

# Los tres ficheros que guardamos en la sesión anterior
enriq_direccion = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_direccion.parquet")
enriq_proximidad = pd.read_parquet(PROCESSED_DIR / "restaurantes_enriquecidos_proximidad.parquet")
censo_rest = pd.read_parquet(PROCESSED_DIR / "censo_restaurantes_madrid.parquet")

print("Ficheros cargados:")
print(f"  Enriquecidos por dirección:  {len(enriq_direccion)} filas × {len(enriq_direccion.columns)} columnas")
print(f"  Enriquecidos por proximidad: {len(enriq_proximidad)} filas × {len(enriq_proximidad.columns)} columnas")
print(f"  Censo restaurantes:          {len(censo_rest)} filas × {len(censo_rest.columns)} columnas")

Ficheros cargados:
  Enriquecidos por dirección:  818 filas × 70 columnas
  Enriquecidos por proximidad: 737 filas × 27 columnas
  Censo restaurantes:          1626 filas × 52 columnas


In [3]:
# =====================================
# INSPECCIONAR COLUMNAS DE CADA FUENTE
# =====================================

print("=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===")
print(list(enriq_direccion.columns))

print("\n=== ENRIQUECIDOS POR PROXIMIDAD (columnas) ===")
print(list(enriq_proximidad.columns))

=== ENRIQUECIDOS POR DIRECCIÓN (columnas) ===
['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine', 'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city', 'phone', 'website', 'opening_hours', 'outdoor_seating', 'takeaway', 'delivery', 'wheelchair', 'calle_norm_osm', 'numero_norm_osm', 'clave_cruce', 'id_local', 'id_distrito_local', 'desc_distrito_local', 'id_barrio_local', 'desc_barrio_local', 'cod_barrio_local', 'id_seccion_censal_local', 'desc_seccion_censal_local', 'coordenada_x_local', 'coordenada_y_local', 'id_tipo_acceso_local', 'desc_tipo_acceso_local', 'id_situacion_local', 'desc_situacion_local', 'id_vial_edificio', 'clase_vial_edificio', 'desc_vial_edificio', 'id_ndp_edificio', 'id_clase_ndp_edificio', 'nom_edificio', 'num_edificio', 'cal_edificio', 'secuencial_local_PC', 'id_vial_acceso', 'clase_vial_acceso', 'desc_vial_acceso', 'id_ndp_acceso', 'id_clase_ndp_acceso', 'nom_acceso', 'num_acceso', 'cal_acceso', 'coordenada_x_agrupacion', 'coordenada_y_agrupacion', 

In [4]:
# =====================================
# UNIFICAR TABLAS ENRIQUECIDAS
# =====================================

# Columnas base de OSM (están en ambas tablas)
cols_osm = ['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine',
            'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city',
            'phone', 'website', 'opening_hours', 'outdoor_seating',
            'takeaway', 'delivery', 'wheelchair']

# Columnas del censo (están en ambas, aunque proximidad tiene menos)
cols_censo = ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
              'id_seccion_censal_local', 'metodo_match']

# La tabla de dirección tiene además el tipo de acceso; la de proximidad no
# Añadimos tipo de acceso solo si existe
cols_direccion = cols_osm + cols_censo
if 'desc_tipo_acceso_local' in enriq_direccion.columns:
    cols_direccion = cols_direccion + ['desc_tipo_acceso_local']

# Preparar cada tabla con las columnas comunes
tabla_dir = enriq_direccion[cols_direccion].copy()

tabla_prox = enriq_proximidad[cols_osm + cols_censo].copy()
tabla_prox['desc_tipo_acceso_local'] = np.nan  # no disponible en proximidad

# Unir las dos
enriquecidos = pd.concat([tabla_dir, tabla_prox], ignore_index=True)

print(f"Total restaurantes enriquecidos unificados: {len(enriquecidos)}")
print(f"  Por dirección:  {(enriquecidos['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad: {(enriquecidos['metodo_match'] == 'proximidad').sum()}")
print(f"\nColumnas: {len(enriquecidos.columns)}")

Total restaurantes enriquecidos unificados: 1555
  Por dirección:  818
  Por proximidad: 737

Columnas: 23


In [5]:
# =====================================
# AÑADIR RESTAURANTES SIN ENRIQUECER
# =====================================

# Cargar el fichero original de OSM (los 1.688 restaurantes)
raw_osm = pd.read_parquet("../data/raw/restaurantes_centro_madrid_osm.parquet")
print(f"Total restaurantes OSM originales: {len(raw_osm)}")

# Identificar los osm_id que YA están enriquecidos
ids_enriquecidos = set(enriquecidos['osm_id'])

# Filtrar los que NO están enriquecidos
sin_enriquecer = raw_osm[~raw_osm['osm_id'].isin(ids_enriquecidos)].copy()
print(f"Restaurantes sin enriquecer: {len(sin_enriquecer)}")

# Añadir las columnas del censo como vacías (no tienen match)
sin_enriquecer['desc_barrio_local'] = np.nan
sin_enriquecer['desc_distrito_local'] = np.nan
sin_enriquecer['desc_epigrafe'] = np.nan
sin_enriquecer['id_seccion_censal_local'] = np.nan
sin_enriquecer['desc_tipo_acceso_local'] = np.nan
sin_enriquecer['metodo_match'] = 'no_matcheado'

# Quedarnos solo con las columnas que tiene la tabla enriquecidos
sin_enriquecer = sin_enriquecer[enriquecidos.columns].copy()

# Unir todo: enriquecidos + sin enriquecer = universo completo
gold_base = pd.concat([enriquecidos, sin_enriquecer], ignore_index=True)

print(f"\n=== UNIVERSO COMPLETO ===")
print(f"Total restaurantes en gold_base: {len(gold_base)}")
print(f"  Por dirección:   {(gold_base['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad:  {(gold_base['metodo_match'] == 'proximidad').sum()}")
print(f"  Sin matchear:    {(gold_base['metodo_match'] == 'no_matcheado').sum()}")

Total restaurantes OSM originales: 1688
Restaurantes sin enriquecer: 331

=== UNIVERSO COMPLETO ===
Total restaurantes en gold_base: 1886
  Por dirección:   818
  Por proximidad:  737
  Sin matchear:    331


In [6]:
# =====================================
# DIAGNÓSTICO DE DUPLICADOS
# =====================================

# ¿Hay osm_id duplicados en la tabla de enriquecidos unificada?
dup_enriq = enriquecidos['osm_id'].duplicated().sum()
print(f"Duplicados dentro de 'enriquecidos': {dup_enriq}")

# ¿Hay solapamiento entre dirección y proximidad?
ids_dir = set(enriq_direccion['osm_id'])
ids_prox = set(enriq_proximidad['osm_id'])
solapamiento = ids_dir & ids_prox
print(f"osm_id que están en AMBAS tablas (dirección y proximidad): {len(solapamiento)}")

# ¿Cuántos osm_id únicos hay en total en enriquecidos?
print(f"osm_id únicos en enriquecidos: {enriquecidos['osm_id'].nunique()} (de {len(enriquecidos)} filas)")

# ¿Duplicados en el raw original?
print(f"osm_id únicos en raw OSM: {raw_osm['osm_id'].nunique()} (de {len(raw_osm)} filas)")

Duplicados dentro de 'enriquecidos': 198
osm_id que están en AMBAS tablas (dirección y proximidad): 0
osm_id únicos en enriquecidos: 1357 (de 1555 filas)
osm_id únicos en raw OSM: 1688 (de 1688 filas)


In [7]:
# =====================================
# ¿DÓNDE ESTÁN LOS DUPLICADOS?
# =====================================

# Duplicados en la tabla de dirección
dup_dir = enriq_direccion['osm_id'].duplicated().sum()
print(f"Duplicados en enriq_direccion: {dup_dir}")

# Duplicados en la tabla de proximidad
dup_prox = enriq_proximidad['osm_id'].duplicated().sum()
print(f"Duplicados en enriq_proximidad: {dup_prox}")

# Ver un ejemplo de restaurante duplicado en dirección
if dup_dir > 0:
    ids_duplicados = enriq_direccion[enriq_direccion['osm_id'].duplicated(keep=False)]['osm_id'].unique()
    ejemplo_id = ids_duplicados[0]
    print(f"\n=== Ejemplo de duplicado (osm_id={ejemplo_id}) ===")
    cols_mostrar = ['osm_id', 'name', 'addr_street', 'rotulo', 'desc_epigrafe', 'desc_barrio_local']
    cols_existentes = [c for c in cols_mostrar if c in enriq_direccion.columns]
    print(enriq_direccion[enriq_direccion['osm_id'] == ejemplo_id][cols_existentes].to_string())

Duplicados en enriq_direccion: 198
Duplicados en enriq_proximidad: 0

=== Ejemplo de duplicado (osm_id=26065699) ===
     osm_id           name          addr_street        rotulo    desc_epigrafe     desc_barrio_local
1  26065699  Honest Greens  Calle de Fuencarral      LA MUCCA  BAR RESTAURANTE  UNIVERSIDAD         
2  26065699  Honest Greens  Calle de Fuencarral    SIN RÓTULO  BAR RESTAURANTE  UNIVERSIDAD         
3  26065699  Honest Greens  Calle de Fuencarral  HONEST GREEN      RESTAURANTE  UNIVERSIDAD         


In [8]:
# =====================================
# DEDUPLICAR: MEJOR MATCH POR SIMILITUD DE NOMBRE
# =====================================

from difflib import SequenceMatcher

def similitud(a, b):
    """Similitud entre dos cadenas (0 a 1)."""
    if pd.isna(a) or pd.isna(b):
        return 0
    return SequenceMatcher(None, str(a).upper(), str(b).upper()).ratio()

# Para la tabla de dirección, calcular similitud entre name (OSM) y rotulo (censo)
enriq_direccion = enriq_direccion.copy()
enriq_direccion['sim_nombre'] = enriq_direccion.apply(
    lambda row: similitud(row['name'], row['rotulo']), axis=1
)

# Para cada osm_id, quedarnos con la fila de mayor similitud
enriq_direccion_dedup = (
    enriq_direccion
    .sort_values('sim_nombre', ascending=False)
    .drop_duplicates(subset='osm_id', keep='first')
    .copy()
)

print(f"Antes de deduplicar: {len(enriq_direccion)} filas")
print(f"Después de deduplicar: {len(enriq_direccion_dedup)} filas")
print(f"osm_id únicos: {enriq_direccion_dedup['osm_id'].nunique()}")

# Ver cómo quedó el ejemplo de Honest Greens
print(f"\n=== Honest Greens tras deduplicar ===")
cols_mostrar = ['osm_id', 'name', 'rotulo', 'desc_epigrafe', 'sim_nombre']
print(enriq_direccion_dedup[enriq_direccion_dedup['osm_id'] == 26065699][cols_mostrar].to_string())

Antes de deduplicar: 818 filas
Después de deduplicar: 620 filas
osm_id únicos: 620

=== Honest Greens tras deduplicar ===
     osm_id           name        rotulo desc_epigrafe  sim_nombre
3  26065699  Honest Greens  HONEST GREEN   RESTAURANTE        0.96


In [9]:
# =====================================
# REHACER UNIFICACIÓN CON TABLA DEDUPLICADA
# =====================================

# Columnas base
cols_osm = ['osm_id', 'osm_type', 'lat', 'lon', 'name', 'cuisine',
            'addr_street', 'addr_housenumber', 'addr_postcode', 'addr_city',
            'phone', 'website', 'opening_hours', 'outdoor_seating',
            'takeaway', 'delivery', 'wheelchair']
cols_censo = ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
              'id_seccion_censal_local', 'metodo_match']

# Tabla dirección deduplicada
cols_dir = cols_osm + cols_censo
if 'desc_tipo_acceso_local' in enriq_direccion_dedup.columns:
    cols_dir = cols_dir + ['desc_tipo_acceso_local']
tabla_dir = enriq_direccion_dedup[cols_dir].copy()

# Tabla proximidad (ya no tiene duplicados)
tabla_prox = enriq_proximidad[cols_osm + cols_censo].copy()
tabla_prox['desc_tipo_acceso_local'] = np.nan

# Unir enriquecidos
enriquecidos = pd.concat([tabla_dir, tabla_prox], ignore_index=True)

# Verificar que no hay solapamiento entre dir y prox
solapamiento = set(tabla_dir['osm_id']) & set(tabla_prox['osm_id'])
print(f"Solapamiento dir/prox: {len(solapamiento)}")

# Añadir los sin enriquecer
ids_enriquecidos = set(enriquecidos['osm_id'])
sin_enriquecer = raw_osm[~raw_osm['osm_id'].isin(ids_enriquecidos)].copy()
for col in ['desc_barrio_local', 'desc_distrito_local', 'desc_epigrafe',
            'id_seccion_censal_local', 'desc_tipo_acceso_local']:
    sin_enriquecer[col] = np.nan
sin_enriquecer['metodo_match'] = 'no_matcheado'
sin_enriquecer = sin_enriquecer[enriquecidos.columns].copy()

# Universo completo
gold_base = pd.concat([enriquecidos, sin_enriquecer], ignore_index=True)

print(f"\n=== UNIVERSO COMPLETO ===")
print(f"Total: {len(gold_base)}")
print(f"osm_id únicos: {gold_base['osm_id'].nunique()}")
print(f"  Por dirección:   {(gold_base['metodo_match'] == 'direccion').sum()}")
print(f"  Por proximidad:  {(gold_base['metodo_match'] == 'proximidad').sum()}")
print(f"  Sin matchear:    {(gold_base['metodo_match'] == 'no_matcheado').sum()}")

Solapamiento dir/prox: 0

=== UNIVERSO COMPLETO ===
Total: 1688
osm_id únicos: 1688
  Por dirección:   620
  Por proximidad:  737
  Sin matchear:    331
